# Notebook 08 — Biomarker Panel & Study Summary

**Project:** Hidden Fingerprints — AI-Powered Comparative Microbiome Analysis  
**Author:** Claire Chien  
**Purpose:** Integrate SHAP, differential abundance, and RF evidence into a composite microbial biomarker panel; generate publication-ready figures and tables; produce a final study summary.

**Inputs (from prior notebooks):**
- `Results/shap_feature_importance.csv`
- `Results/shap_per_class_top_genera.csv`
- `Results/diff_abundance_colorectal.csv`
- `Results/diff_abundance_breast.csv`
- `Results/diff_abundance_crosscancer.csv`
- `Results/pancancer_shared_enriched_genera.csv`
- `Results/model_comparison.csv`
- `Results/rf_feature_importances.csv`
- `Results/meta_combined.csv`

**Outputs:**
- `Figures/fig13_biomarker_panel.png`
- `Figures/fig01_study_design.png`
- `Results/Table3_biomarker_panel.csv`

In [ ]:
# Cell 1 — Imports
import pandas as pd         # pandas: tables and data manipulation (like Excel in Python)
import numpy as np          # numpy: fast math on arrays of numbers
import matplotlib           # matplotlib: core Python charting library
matplotlib.use('Agg')       # 'Agg' = save plots to files instead of opening pop-up windows
import matplotlib.pyplot as plt         # pyplot: easier interface for drawing charts
import matplotlib.patches as mpatches   # mpatches: shapes like rectangles and circles for diagrams
import matplotlib.gridspec as gridspec  # gridspec: custom layout grid for multi-panel figures
import seaborn as sns       # seaborn: prettier statistical charts built on matplotlib
import warnings             # warnings: controls Python's warning messages
warnings.filterwarnings('ignore')  # suppress harmless warnings to keep output clean
import os                   # os: file and folder path utilities

print('Imports complete.')
print(f'pandas {pd.__version__}, numpy {np.__version__}, matplotlib {matplotlib.__version__}')

In [ ]:
# Cell 2 — Paths and Load Results from Prior Notebooks
# This notebook is the final assembly step — it reads all tables saved by NB03–NB07
# and combines them into a single ranked biomarker panel

# Build the base directory path by going up one level from the Notebooks/ folder
BASE    = os.path.abspath(os.path.join(os.path.dirname('__file__') if '__file__' in dir() else '.', '..'))
RESULTS = os.path.join(BASE, 'Results')   # tables saved by prior notebooks live here
FIGURES = os.path.join(BASE, 'Figures')   # chart images will be saved here

os.makedirs(FIGURES, exist_ok=True)  # create folder if it doesn't already exist
os.makedirs(RESULTS, exist_ok=True)

# safe_load() — a helper function that loads a CSV but returns an empty table if the
# file doesn't exist yet. This lets the notebook run even if some prior notebook failed.
# "Best-effort" loading: warn the user instead of crashing with an error.
def safe_load(path, **kwargs):
    try:
        df = pd.read_csv(path, **kwargs)  # read the CSV; **kwargs passes extra options through
        print(f'  Loaded: {os.path.basename(path):50s} -> {df.shape}')
        return df
    except FileNotFoundError:
        print(f'  Warning: {path} not found. Run prior notebooks first.')
        return pd.DataFrame()  # empty DataFrame so downstream code doesn't crash

print('Loading results CSVs ...')

# SHAP global importance (from NB07): one row per bacterium, with mean |SHAP| score
shap_df       = safe_load(os.path.join(RESULTS, 'shap_feature_importance.csv'))

# SHAP per-class top 10 (from NB07): top bacteria per cancer type
shap_class_df = safe_load(os.path.join(RESULTS, 'shap_per_class_top_genera.csv'))

# Differential abundance results (from NB05): Mann-Whitney p-values and fold changes
da_crc_df     = safe_load(os.path.join(RESULTS, 'diff_abundance_colorectal.csv'))
da_breast_df  = safe_load(os.path.join(RESULTS, 'diff_abundance_breast.csv'))

# Cross-cancer Kruskal-Wallis results (from NB05): which bacteria differ across all 3 types
da_cross_df   = safe_load(os.path.join(RESULTS, 'diff_abundance_crosscancer.csv'))

# Pan-cancer genera: bacteria significantly enriched in BOTH colorectal AND breast cancer
pancancer_df  = safe_load(os.path.join(RESULTS, 'pancancer_shared_enriched_genera.csv'))

# ML model comparison table (from NB06): AUC, accuracy, F1 for each model
model_comp_df = safe_load(os.path.join(RESULTS, 'model_comparison.csv'))

# Random Forest feature importances (from NB06): how much each genus helps RF classify
rf_imp_df     = safe_load(os.path.join(RESULTS, 'rf_feature_importances.csv'))

# Combined metadata: sample IDs with cancer_type and condition labels
meta_df       = safe_load(os.path.join(RESULTS, 'meta_combined.csv'))

print('\nAll files processed.')

In [ ]:
# Cell 3 — Build Unified Biomarker Composite Scoring
# We combine four independent lines of evidence into one number per bacterium:
#
#   composite_score = 0.4 × shap_score       (SHAP: ML model explanation)
#                   + 0.3 × da_evidence      (DA: statistically enriched in cancer)
#                   + 0.2 × cross_cancer     (significant across ALL 3 cancer types)
#                   + 0.1 × rf_importance    (Random Forest importance as backup)
#
# The weights (0.4, 0.3, 0.2, 0.1) reflect how much we trust each evidence source.
# SHAP is given the most weight because it directly explains the best ML model.
# The final score ranges from 0 (no evidence) to 1 (highest evidence across all methods).

def normalise_series(s):
    """Min-max normalise a pandas Series to [0, 1].
    Min-max: (value - minimum) / (maximum - minimum)
    The smallest value becomes 0, the largest becomes 1, others scale in between.
    """
    rng = s.max() - s.min()  # range = max minus min
    if rng == 0:
        return pd.Series(np.zeros(len(s)), index=s.index)  # all same value → all 0
    return (s - s.min()) / rng

# ---- Step 1: SHAP scores -----------------------------------
# Use the global mean |SHAP| importance computed in Notebook 07
if not shap_df.empty:
    shap_col      = 'mean_abs_shap' if 'mean_abs_shap' in shap_df.columns else shap_df.columns[1]
    genus_col_shap = 'genus' if 'genus' in shap_df.columns else shap_df.columns[0]
    shap_base     = shap_df[[genus_col_shap, shap_col]].copy()
    shap_base.columns = ['genus', 'raw_shap']
    shap_base['shap_score'] = normalise_series(shap_base['raw_shap'])  # scale to 0–1
else:
    # Synthetic fallback: if SHAP file is missing, create dummy values so the notebook runs
    fallback_genera = [
        'Fusobacterium', 'Prevotella', 'Bacteroides', 'Peptostreptococcus',
        'Porphyromonas', 'Streptococcus', 'Lactobacillus', 'Bifidobacterium',
        'Akkermansia', 'Faecalibacterium', 'Clostridium', 'Ruminococcus',
        'Cutibacterium', 'Methylobacterium', 'Sphingomonas', 'Veillonella',
        'Alistipes', 'Lachnospiraceae_NK4A136', 'Blautia', 'Roseburia'
    ]
    np.random.seed(42)  # seed for reproducibility
    raw_vals  = np.sort(np.random.exponential(0.05, size=len(fallback_genera)))[::-1]
    shap_base = pd.DataFrame({'genus': fallback_genera, 'raw_shap': raw_vals})
    shap_base['shap_score'] = normalise_series(shap_base['raw_shap'])
    print('Warning: Using synthetic SHAP scores (shap_feature_importance.csv not found).')

# ---- Step 2: Differential abundance evidence ---------------
# A bacterium gets da_evidence = 1 if it was statistically enriched in ANY cancer type
# 0 = not significantly different in any comparison (adj_p >= 0.05)
sig_genera_crc    = set()  # genera enriched in colorectal cancer
sig_genera_breast = set()  # genera enriched in breast cancer

if not da_crc_df.empty:
    g_col = 'genus' if 'genus' in da_crc_df.columns else da_crc_df.columns[0]
    p_col = 'adjusted_p' if 'adjusted_p' in da_crc_df.columns else 'p_adj'
    if p_col in da_crc_df.columns:
        sig_genera_crc = set(da_crc_df.loc[da_crc_df[p_col] < 0.05, g_col].dropna())

if not da_breast_df.empty:
    g_col = 'genus' if 'genus' in da_breast_df.columns else da_breast_df.columns[0]
    p_col = 'adjusted_p' if 'adjusted_p' in da_breast_df.columns else 'p_adj'
    if p_col in da_breast_df.columns:
        sig_genera_breast = set(da_breast_df.loc[da_breast_df[p_col] < 0.05, g_col].dropna())

all_sig_da = sig_genera_crc | sig_genera_breast  # union: enriched in CRC OR breast OR both

# ---- Step 3: Cross-cancer significance ---------------------
# A bacterium gets cross_cancer = 1 if Kruskal-Wallis shows it differs across all 3 cancer types
# This is strong evidence that the bacterium is consistently linked to multiple cancers
cross_sig_genera = set()
if not da_cross_df.empty:
    g_col = 'genus' if 'genus' in da_cross_df.columns else da_cross_df.columns[0]
    p_col = next((c for c in ['adjusted_p', 'p_adj', 'p_value'] if c in da_cross_df.columns), None)
    if p_col:
        cross_sig_genera = set(da_cross_df.loc[da_cross_df[p_col] < 0.05, g_col].dropna())

# ---- Step 4: Random Forest importance ----------------------
# Normalized RF feature importance (0–1) — how useful each genus was to the RF model
if not rf_imp_df.empty:
    g_col = 'genus' if 'genus' in rf_imp_df.columns else rf_imp_df.columns[0]
    v_col = next((c for c in ['importance', 'mean_importance', 'feature_importance']
                  if c in rf_imp_df.columns), rf_imp_df.columns[-1])
    rf_base = rf_imp_df[[g_col, v_col]].copy()
    rf_base.columns = ['genus', 'rf_raw']
    rf_base['rf_importance'] = normalise_series(rf_base['rf_raw'])  # scale to 0–1
else:
    # Fallback: use the SHAP genus list with random RF importances
    rf_base = shap_base[['genus']].copy()
    rf_base['rf_importance'] = normalise_series(
        pd.Series(np.random.exponential(0.03, size=len(rf_base)), index=rf_base.index)
    )

# ---- Build the unified biomarker DataFrame -----------------
# Start with SHAP scores as the base (259 genera)
biomarker_df = shap_base[['genus', 'shap_score']].copy()

# Add binary columns: 1 if the genus is significant, 0 otherwise
biomarker_df['da_evidence']  = biomarker_df['genus'].apply(lambda g: 1 if g in all_sig_da else 0)
biomarker_df['cross_cancer'] = biomarker_df['genus'].apply(lambda g: 1 if g in cross_sig_genera else 0)

# Merge in RF importance — if a genus is not in the RF top-50, fill with 0
biomarker_df = biomarker_df.merge(rf_base[['genus', 'rf_importance']], on='genus', how='left')
biomarker_df['rf_importance'] = biomarker_df['rf_importance'].fillna(0)

# Compute the composite score: weighted sum of all four evidence sources
biomarker_df['composite_score'] = (
    0.4 * biomarker_df['shap_score']    +  # SHAP: 40% weight
    0.3 * biomarker_df['da_evidence']   +  # DA significance: 30% weight
    0.2 * biomarker_df['cross_cancer']  +  # Cross-cancer KW: 20% weight
    0.1 * biomarker_df['rf_importance']    # RF importance: 10% weight
)

# Determine the cancer type this genus is most associated with
# Use SHAP per-class data if available; otherwise infer from DA significance
if not shap_class_df.empty and 'cancer_type' in shap_class_df.columns:
    g_col = 'genus' if 'genus' in shap_class_df.columns else shap_class_df.columns[0]
    v_col = 'mean_abs_shap' if 'mean_abs_shap' in shap_class_df.columns else shap_class_df.columns[2]
    # For each genus, find the cancer class with the highest SHAP value
    primary = (
        shap_class_df.groupby(g_col)
        .apply(lambda x: x.loc[x[v_col].idxmax(), 'cancer_type'])
        .reset_index()
    )
    primary.columns = ['genus', 'primary_cancer']
    biomarker_df = biomarker_df.merge(primary, on='genus', how='left')
else:
    # Fallback: assign primary cancer based on which DA test was significant
    def infer_primary(g):
        if g in sig_genera_crc and g in sig_genera_breast:
            return 'Pan-cancer'  # significant in both → not cancer-type specific
        elif g in sig_genera_crc:
            return 'Colorectal'
        elif g in sig_genera_breast:
            return 'Breast'
        else:
            return 'Unassigned'  # not DA-significant in any comparison
    biomarker_df['primary_cancer'] = biomarker_df['genus'].apply(infer_primary)

# Sort by composite score descending (best biomarker candidates at the top)
biomarker_df = biomarker_df.sort_values('composite_score', ascending=False).reset_index(drop=True)

print('Biomarker dataframe built.')
print(f'Total genera scored: {len(biomarker_df)}')
print(f'\nTop 10 by composite score:')
print(biomarker_df[['genus', 'composite_score', 'primary_cancer']].head(10).to_string(index=False))

In [ ]:
# Cell 4 — Annotate Biomarkers with Known Biological Roles
# For each top bacterium, we add a short description of what science already knows
# about its connection to cancer. This turns numbers into biological meaning.
# Genera NOT in this dictionary are marked as "Novel candidates" — they may be
# important but haven't been studied in this context yet, so they warrant new research.

KNOWN_ROLES = {
    'Fusobacterium':            'Colorectal cancer promoter; promotes tumor invasion via FadA adhesin',
    'Prevotella':               'Associated with colorectal cancer; linked to mucosal inflammation',
    'Bacteroides':              'Commensal with complex roles; some species protective, others oncogenic',
    'Peptostreptococcus':       'Enriched in colorectal cancer; activates oncogenic signaling',
    'Porphyromonas':            'Periodontal pathogen; found in colorectal and breast tumors',
    'Streptococcus':            'Associated with multiple cancer types; promotes biofilm formation',
    'Lactobacillus':            'Generally protective; produces bacteriocins; reduced in some cancers',
    'Bifidobacterium':          'Probiotic genus; often depleted in cancer patients',
    'Akkermansia':              'Mucin-degrader; complex role; often depleted in cancer',
    'Faecalibacterium':         'Anti-inflammatory butyrate producer; depleted in colorectal cancer',
    'Clostridium':              'Diverse genus; some species oncogenic via secondary bile acids',
    'Lachnospiraceae':          'SCFA producers; often depleted in colorectal cancer',
    'Lachnospiraceae_NK4A136':  'SCFA producers; often depleted in colorectal cancer',
    'Ruminococcus':             'Fiber fermenter; reduced in colorectal cancer patients',
    'Cutibacterium':            'Skin commensal found in prostate tissue; emerging cancer association',
    'Methylobacterium':         'Found in breast tumor microenvironment; promotes tumor growth',
    'Sphingomonas':             'Environmental bacterium found in breast tumors',
    'Veillonella':              'Oral bacterium; found elevated in multiple cancers',
    'Alistipes':                'Mucosal bacterium; complex roles in cancer inflammation',
    'Blautia':                  'Butyrate producer; depleted in colorectal cancer; modulates immunity',
    'Roseburia':                'Butyrate producer; often reduced in colorectal cancer',
    'Peptoniphilus':            'Anaerobic gram-positive; found in breast cancer tissue microbiome',
    'Corynebacterium':          'Skin commensal; elevated in some prostate cancer studies',
    'Finegoldia':               'Anaerobic gram-positive; detected in breast and prostate tumors',
    'Anaerococcus':             'Anaerobic gram-positive coccus; found in breast tumor microbiome',
}

# Label for bacteria not found in our literature dictionary
NOVEL_LABEL = 'Novel candidate — literature review needed'

# .apply() runs the lambda function on every genus name in the column
# KNOWN_ROLES.get(g, NOVEL_LABEL) looks up g in the dictionary;
# if not found, returns NOVEL_LABEL as the default
biomarker_df['known_role'] = biomarker_df['genus'].apply(
    lambda g: KNOWN_ROLES.get(g, NOVEL_LABEL)
)

top20 = biomarker_df.head(20).copy()  # look at the top 20 biomarker candidates

# Count how many of the top 20 have existing literature evidence
known_count = (top20['known_role'] != NOVEL_LABEL).sum()  # .sum() counts True values

print(f'Top 20 genera annotated: {known_count} with known roles, '
      f'{20 - known_count} novel candidates.')
print('\nAnnotated top 20:')
print(top20[['genus', 'composite_score', 'primary_cancer', 'known_role']]
      .to_string(index=False))

In [ ]:
# Cell 5 — Figure 13: Biomarker Panel Heatmap
# This figure shows the top 15 biomarker candidates as a color-coded heatmap:
#   - Each row = one bacterial genus (ranked 1 = best evidence)
#   - Each column = one evidence source (SHAP, DA, cross-cancer, RF importance, composite)
#   - Color: yellow (low) → orange → red (high evidence)
#   - Left strip: cancer type color and rank number
#   - Right panel: known biological role description
# This is "Figure 13" — the final synthesis figure of the entire project

top15 = biomarker_df.head(15).copy()  # top 15 biomarker candidates
top15['rank'] = range(1, 16)          # add rank numbers 1–15

# The columns to display in the heatmap grid
score_cols  = ['composite_score', 'shap_score', 'da_evidence', 'cross_cancer', 'rf_importance']
col_labels  = ['Composite\nScore', 'SHAP\nEvidence', 'DA\nEvidence', 'Cross-\nCancer', 'RF\nImportance']

# Assign a distinct color to each cancer type for the left-side colored strip
cancer_colors = {
    'Colorectal':  '#e74c3c',  # red
    'Breast':      '#9b59b6',  # purple
    'Prostate':    '#3498db',  # blue
    'Pan-cancer':  '#e67e22',  # orange
    'Unassigned':  '#95a5a6',  # grey
}

# Create a figure with 3 side-by-side panels using GridSpec for precise layout control
# GridSpec lets us specify different widths for each panel (like column-width in a table)
fig = plt.figure(figsize=(20, 11))
gs  = gridspec.GridSpec(1, 3, figure=fig,
                        width_ratios=[0.3, 2.5, 1.8],  # narrow | wide heatmap | wide legend
                        wspace=0.05)                    # tiny gap between panels

ax_rank   = fig.add_subplot(gs[0])  # left panel: rank numbers + cancer type colors
ax_main   = fig.add_subplot(gs[1])  # center panel: the actual heatmap
ax_legend = fig.add_subplot(gs[2])  # right panel: known role text annotations

# ---- Left panel: rank strip and cancer-type colored boxes ---
ax_rank.set_xlim(0, 1)    # coordinate space for placing shapes manually
ax_rank.set_ylim(-0.5, 14.5)
ax_rank.invert_yaxis()    # rank 1 at top (y=0), rank 15 at bottom (y=14)
ax_rank.axis('off')       # hide x/y axes — we draw everything manually

for i, row in top15.iterrows():
    rank_i = row['rank']
    y_pos  = rank_i - 1  # y coordinate for this row (0-indexed)

    # Get the color for this genus's primary cancer type
    cc = cancer_colors.get(str(row.get('primary_cancer', 'Unassigned')), '#95a5a6')

    # Draw a colored rounded rectangle on the right half of this panel
    ax_rank.add_patch(mpatches.FancyBboxPatch(
        (0.55, y_pos - 0.42), 0.40, 0.84,          # (x, y), width, height
        boxstyle='round,pad=0.04',                  # slightly rounded corners
        facecolor=cc, alpha=0.85, edgecolor='white' # cancer-type color, slightly transparent
    ))
    # Print the rank number (left side)
    ax_rank.text(0.27, y_pos, f'{rank_i}',
                 ha='center', va='center', fontsize=11, fontweight='bold', color='#2c3e50')
    # Print the first letter of the cancer type in the colored box
    ax_rank.text(0.75, y_pos, str(row.get('primary_cancer', 'N/A'))[0],
                 ha='center', va='center', fontsize=8, fontweight='bold', color='white')

ax_rank.set_title('Rank\n& Type', fontsize=9, color='#555', pad=4)

# ---- Center panel: heatmap ----------------------------------
# .values converts the DataFrame columns to a 2D NumPy array for imshow
heat_data    = top15[score_cols].values
genus_labels = [f'{g}' for g in top15['genus']]

# imshow displays a 2D array as a color image
# cmap='YlOrRd' = Yellow-Orange-Red color scale (low = yellow, high = red)
# vmin/vmax = 0/1 anchors the color scale to our normalized [0,1] range
im = ax_main.imshow(heat_data, aspect='auto', cmap='YlOrRd',
                    vmin=0, vmax=1, interpolation='nearest')

ax_main.set_xticks(range(len(score_cols)))
ax_main.set_xticklabels(col_labels, fontsize=10)
ax_main.set_yticks(range(15))
ax_main.set_yticklabels(genus_labels, fontsize=11, style='italic')  # genus names in italics
ax_main.tick_params(axis='x', top=True, bottom=False,
                    labeltop=True, labelbottom=False)  # put column labels at the TOP
ax_main.tick_params(axis='y', left=True)

# Add numeric score labels inside each cell
for r in range(heat_data.shape[0]):       # for each row (genus)
    for c in range(heat_data.shape[1]):   # for each column (evidence type)
        val  = heat_data[r, c]
        text = f'{val:.2f}'               # show 2 decimal places
        col  = 'white' if val > 0.6 else '#2c3e50'  # white text on dark cells, dark on light
        ax_main.text(c, r, text, ha='center', va='center',
                     fontsize=9, color=col, fontweight='bold')

# Draw white grid lines between cells to make them visually distinct
for r in range(16):  # 16 horizontal lines for 15 rows
    ax_main.axhline(r - 0.5, color='white', lw=0.8)
for c in range(6):   # 6 vertical lines for 5 columns
    ax_main.axvline(c - 0.5, color='white', lw=0.8)

ax_main.set_title('Proposed Microbial Biomarker Panel\n(Multi-Evidence Composite Scoring)',
                  fontsize=13, fontweight='bold', pad=18)

# Add a colorbar (legend for the color scale) below the heatmap
cbar = fig.colorbar(im, ax=ax_main, orientation='horizontal',
                    fraction=0.03, pad=0.12, shrink=0.6)
cbar.set_label('Normalised Evidence Score (0–1)', fontsize=9)

# ---- Right panel: biological role annotations ---------------
ax_legend.axis('off')    # hide axes — we position text manually
ax_legend.set_xlim(0, 1)
ax_legend.set_ylim(0, 1)

ax_legend.text(0.0, 0.98, 'Known Biological Role',
               fontsize=10, fontweight='bold', color='#2c3e50', va='top', ha='left')

row_h = 0.84 / 15  # vertical height allocated per row

for idx, (_, row) in enumerate(top15.iterrows()):
    y_top = 0.92 - idx * row_h  # y position for this genus's role text
    role  = str(row['known_role'])
    if len(role) > 62:
        role = role[:59] + '...'  # truncate very long descriptions to fit the panel

    bg = '#fdf6e3' if idx % 2 == 0 else '#ffffff'  # alternating row background colors
    ax_legend.add_patch(mpatches.Rectangle(
        (0, y_top - row_h * 0.9), 1.0, row_h * 0.95,
        transform=ax_legend.transAxes,
        facecolor=bg, edgecolor='#ddd', lw=0.4
    ))
    ax_legend.text(0.03, y_top - row_h * 0.38, role,
                   fontsize=7, va='center', ha='left', color='#444', wrap=True)

# Add a cancer-type color legend below the heatmap
legend_patches = [mpatches.Patch(facecolor=v, label=k) for k, v in cancer_colors.items()]
ax_main.legend(handles=legend_patches, title='Primary Cancer',
               loc='lower right', fontsize=8, title_fontsize=8,
               framealpha=0.9, bbox_to_anchor=(1.0, -0.22))

fig.suptitle('Figure 13 — Microbial Biomarker Panel for Multi-Cancer Classification',
             fontsize=14, fontweight='bold', y=1.01)

out13 = os.path.join(FIGURES, 'fig13_biomarker_panel.png')
plt.savefig(out13, dpi=300, bbox_inches='tight', facecolor='white')  # save at high resolution
plt.close()
print(f'Saved: {out13}')

In [ ]:
# Cell 6 — Table 3: Final Publication-Ready Biomarker Table
# This table summarizes the top 15 biomarker candidates for the scientific paper.
# It combines all evidence sources into one easy-to-read table with:
#   - Rank: 1 = strongest combined evidence
#   - Composite score: the weighted average of all four evidence sources
#   - Primary cancer association: which cancer type this genus is most linked to
#   - DA significance: whether this genus passed statistical thresholds (adj_p < 0.05)
#   - SHAP rank: position in the global SHAP importance ranking
#   - Known biological role: brief literature annotation (or "Novel candidate")

top15 = biomarker_df.head(15).copy().reset_index(drop=True)  # top 15, re-indexed from 0

# Build a lookup dictionary: genus name → its SHAP rank (1 = most important to ML model)
shap_rank_map = {g: i + 1 for i, g in enumerate(shap_base['genus'].tolist())}
top15['shap_rank'] = top15['genus'].map(shap_rank_map).fillna('N/A')  # N/A if genus not in SHAP

# Build a human-readable differential abundance significance label for each genus
def da_label(g):
    labels = []
    if g in sig_genera_crc:
        labels.append('CRC p<0.05')           # significant in colorectal cancer DA
    if g in sig_genera_breast:
        labels.append('Breast p<0.05')        # significant in breast cancer DA
    if g in cross_sig_genera:
        labels.append('Cross-cancer p<0.05')  # significant in cross-cancer KW test
    return '; '.join(labels) if labels else 'Not significant'  # join with semicolons

top15['da_significance'] = top15['genus'].apply(da_label)

# Assemble the final Table 3 with clear, meaningful column names
table3 = pd.DataFrame({
    'Rank'                  : range(1, 16),
    'Genus'                 : top15['genus'],
    'Composite_Score'       : top15['composite_score'].round(3),   # round to 3 decimal places
    'Primary_Cancer_Assoc'  : top15['primary_cancer'].fillna('Unassigned'),
    'DA_Significance'       : top15['da_significance'],
    'SHAP_Rank'             : top15['shap_rank'],
    'Known_Biological_Role' : top15['known_role'],
})

out_table = os.path.join(RESULTS, 'Table3_biomarker_panel.csv')
table3.to_csv(out_table, index=False)  # save without row numbers

# Display the table in the notebook output
print('Table 3 — Final Biomarker Panel')
print('=' * 120)
pd.set_option('display.max_colwidth', 55)  # wrap long text in the 'Known Role' column
pd.set_option('display.width', 200)         # use more screen width before wrapping
print(table3.to_string(index=False))
print(f'\nSaved: {out_table}')

In [ ]:
# Cell 7 — Pan-Cancer Microbial Signature Summary
# "Pan-cancer" means bacteria that are enriched in MULTIPLE cancer types simultaneously
# (in this study: enriched in both Colorectal AND Breast cancer vs healthy controls)
# These are especially interesting because they might reflect shared mechanisms that
# drive cancer in general — not just tissue-specific effects.
# For example, bacteria that cause chronic inflammation could promote cancer in many organs.

# Try to get pan-cancer genera from multiple sources (in priority order):
pancancer_genera = []

if not pancancer_df.empty:
    # Source 1: dedicated file saved by Notebook 05 (set intersection of enriched genera)
    g_col = 'genus' if 'genus' in pancancer_df.columns else pancancer_df.columns[0]
    pancancer_genera = pancancer_df[g_col].dropna().tolist()

elif len(cross_sig_genera) > 0:
    # Source 2: genera significant across all 3 cancer types in the Kruskal-Wallis test
    pancancer_genera = sorted(cross_sig_genera)

else:
    # Source 3: manual intersection of CRC and breast significant genera
    pancancer_genera = sorted(sig_genera_crc & sig_genera_breast) if sig_genera_crc and sig_genera_breast else []

n_pancancer = len(pancancer_genera)  # how many pan-cancer genera did we find?

# Format the first 10 genera as a comma-separated string for the summary text
genera_str  = ', '.join([f'\'{g}\'' for g in pancancer_genera[:10]])
if n_pancancer > 10:
    genera_str += f' ... and {n_pancancer - 10} more'  # indicate there are more beyond the first 10

print('=' * 78)
print('PAN-CANCER MICROBIAL SIGNATURE SUMMARY')
print('=' * 78)

if n_pancancer > 0:
    summary = (
        f'{n_pancancer} genera were found enriched across both Colorectal and Breast '
        f'cancers compared to healthy controls. These pan-cancer microbial signatures '
        f'include: {genera_str}.\n\n'
        f'Pan-cancer genera are particularly important because they may reflect shared '
        f'oncogenic mechanisms — such as immune evasion, pro-inflammatory signalling, '
        f'or altered metabolite production — rather than tissue-specific colonisation. '
        f'These candidates should be prioritised for validation in independent cohorts '
        f'and mechanistic in vitro studies.'
    )
else:
    summary = (
        'No pan-cancer genera file was found or no genera reached significance thresholds '
        'in both colorectal and breast cancer datasets. Re-run prior notebooks (05–07) to '
        'generate pancancer_shared_enriched_genera.csv.'
    )

print(summary)
print('=' * 78)

In [ ]:
# Cell 8 — Figure 1: Study Design Overview (Flow Diagram)
# This figure shows the entire project pipeline as a diagram:
#   Top: the three cancer datasets (Colorectal, Breast, Prostate)
#   Middle: the five analysis steps as a left-to-right flow
#   Bottom: the final outputs (biomarker panel, paper, Table 3)
# Scientists call this a "study design figure" — it lets readers understand the
# whole project in one glance before reading any text.
# We draw it entirely with matplotlib shapes (no special diagram library needed).

fig, ax = plt.subplots(figsize=(16, 9))
ax.set_xlim(0, 16)   # custom coordinate system: 0–16 wide
ax.set_ylim(0, 9)    # 0–9 tall
ax.axis('off')       # hide the default x/y axes — we draw everything manually

# ---- Color palette -----------------------------------------
COLORS = {
    'data':      '#2980b9',  # blue — data source boxes
    'qc':        '#27ae60',  # green — QC step
    'eda':       '#8e44ad',  # purple — diversity analysis
    'ml':        '#e74c3c',  # red — ML classification
    'shap':      '#e67e22',  # orange — SHAP interpretability
    'biomarker': '#16a085',  # teal — final biomarker output
    'crc':       '#c0392b',  # dark red — colorectal cancer label
    'breast':    '#8e44ad',  # purple — breast cancer label
    'prostate':  '#2980b9',  # blue — prostate cancer label
    'arrow':     '#555',     # dark grey — arrows
    'bg':        '#f8f9fa',  # very light grey — background
}

fig.patch.set_facecolor(COLORS['bg'])  # set the whole figure background
ax.set_facecolor(COLORS['bg'])

# draw_box() — helper to draw a labeled rounded rectangle at (x, y) with given width/height
def draw_box(ax, x, y, w, h, label, sublabel, color, fontsize=10):
    box = mpatches.FancyBboxPatch(
        (x - w/2, y - h/2), w, h,           # centered at (x, y)
        boxstyle='round,pad=0.15',            # slightly rounded corners
        facecolor=color, edgecolor='white',
        linewidth=2, alpha=0.92, zorder=3    # zorder=3: drawn on top of arrows
    )
    ax.add_patch(box)
    ax.text(x, y + 0.15, label,              # main label text (slightly above center)
            ha='center', va='center', fontsize=fontsize,
            fontweight='bold', color='white', zorder=4)
    ax.text(x, y - 0.30, sublabel,           # subtitle text (slightly below center)
            ha='center', va='center', fontsize=7.5,
            color='white', alpha=0.90, zorder=4)

# draw_arrow() — helper to draw a horizontal arrow from x1 to x2 at height y
def draw_arrow(ax, x1, x2, y, color='#555'):
    ax.annotate('',
                xy=(x2 - 0.05, y), xytext=(x1 + 0.05, y),  # start and end of arrow
                arrowprops=dict(arrowstyle='->', color=color,
                                lw=2.0, mutation_scale=20),  # arrowhead size
                zorder=2)

# ---- Title and subtitle ------------------------------------
ax.text(8, 8.55, 'Hidden Fingerprints — Study Design Overview',
        ha='center', va='center', fontsize=15, fontweight='bold', color='#2c3e50')
ax.text(8, 8.10, 'AI-Powered Comparative Microbiome Analysis: Colorectal | Breast | Prostate Cancer',
        ha='center', va='center', fontsize=10, color='#555')

# ---- Three cancer type circles at the top ------------------
# Each circle represents one of the three cancer datasets feeding into our pipeline
for cx, clabel, ccol in [
    (3.2,  'Colorectal\nCancer', COLORS['crc']),
    (8.0,  'Breast\nCancer',     COLORS['breast']),
    (12.8, 'Prostate\nCancer',   COLORS['prostate']),
]:
    circ = plt.Circle((cx, 6.9), 0.65, color=ccol, zorder=3, alpha=0.90)
    ax.add_patch(circ)
    ax.text(cx, 6.9, clabel, ha='center', va='center',
            fontsize=7.5, fontweight='bold', color='white', zorder=4)

# ---- Data source boxes (NCBI accession numbers) ------------
# Below each circle: where the raw sequencing data came from (public databases)
for dx, dlabel, dsub in [
    (3.2,  'NCBI SRA\nPRJNA290926',  'CRC — stool 16S'),
    (8.0,  'NCBI SRA\nPRJNA658160',  'Breast — tissue 16S'),
    (12.8, 'NCBI SRA\nPRJNA1298576', 'Prostate — urine 16S'),
]:
    draw_box(ax, dx, 5.75, 2.5, 0.80, dlabel, dsub, COLORS['data'], fontsize=8.5)
    ax.annotate('', xy=(dx, 6.22), xytext=(dx, 6.25),  # short arrow from circle to box
                arrowprops=dict(arrowstyle='->', color=COLORS['arrow'],
                                lw=1.4, mutation_scale=16), zorder=2)

# ---- Converging dashed lines from three datasets to one pipeline ----
# The three separate datasets merge into one unified analysis pipeline
for sx in [3.2, 8.0, 12.8]:
    ax.plot([sx, 8.0], [5.35, 4.85], color=COLORS['arrow'],
            lw=1.5, ls='--', zorder=2, alpha=0.7)  # dashed line: ls='--'

# ---- Five pipeline analysis steps (horizontal flow) --------
pipeline_steps = [
    (2.5,  'QC &\nHarmonization',    'Prevalence filtering\nCLR transform\nGenus-level union',  COLORS['qc']),
    (5.5,  'EDA &\nDiversity',        'Alpha / Beta diversity\nPCoA · UMAP\nKruskal-Wallis',     COLORS['eda']),
    (8.5,  'Differential\nAbundance', 'Mann-Whitney U\nFDR (BH)\n|log2FC| > 0.5',              COLORS['data']),
    (11.5, 'ML\nClassification',      'RF · XGBoost\nLogReg · SVM\n5-fold CV · SMOTE',         COLORS['ml']),
    (14.5, 'SHAP\nInterpretability',  'TreeExplainer\nPer-class SHAP\nTop genera',              COLORS['shap']),
]

for px, plabel, psub, pcol in pipeline_steps:
    draw_box(ax, px, 3.5, 2.6, 1.50, plabel, psub, pcol, fontsize=9)

# Draw horizontal arrows connecting the five steps left to right
for i in range(len(pipeline_steps) - 1):
    x1 = pipeline_steps[i][0]   + 1.30   # right edge of current step box
    x2 = pipeline_steps[i+1][0] - 1.30   # left edge of next step box
    draw_arrow(ax, x1, x2, 3.5)

# Arrow from where the three datasets converge into the first pipeline step (QC)
ax.annotate('', xy=(2.5, 4.25), xytext=(8.0, 4.85),
            arrowprops=dict(arrowstyle='->', color=COLORS['arrow'],
                            lw=1.4, mutation_scale=16), zorder=2)

# ---- Final output box at the bottom ------------------------
draw_box(ax, 8.0, 1.5, 8.0, 1.20,
         'Biomarker Panel & Study Outputs',
         'Table 3: 15 composite-scored genera  |  Fig 13: Biomarker heatmap  |  '
         'Paper: draft_v1.md  |  Pan-cancer signature report',
         COLORS['biomarker'], fontsize=10)

# Arrow from SHAP (last step) down to the output box
ax.annotate('', xy=(8.0, 2.10), xytext=(14.5, 2.75),
            arrowprops=dict(arrowstyle='->', color=COLORS['arrow'],
                            lw=1.6, mutation_scale=18), zorder=2)

# ---- Step numbers below each pipeline box ------------------
for step_n, (px, _, _, _) in enumerate(pipeline_steps, start=1):
    ax.text(px, 2.65, f'Step {step_n}',
            ha='center', va='center', fontsize=8, color='#777', fontstyle='italic')

out01 = os.path.join(FIGURES, 'fig01_study_design.png')
plt.savefig(out01, dpi=300, bbox_inches='tight', facecolor=COLORS['bg'])  # high-res save
plt.close()
print(f'Saved: {out01}')

In [ ]:
# Cell 9 — Final Project Summary Report
# This cell prints a human-readable summary of the ENTIRE project:
#   [1] Dataset sizes — how many samples per cancer type
#   [2] Best ML model performance (AUC, accuracy, F1)
#   [3] Top 5 microbial biomarkers by composite score
#   [4] Pan-cancer microbial signatures
#   [5] Key diversity findings from alpha diversity analysis
#   [6] Status of all output files generated by this notebook
# Think of this as the "executive summary" that a scientist would read first

print('\n' + '=' * 78)
print('  HIDDEN FINGERPRINTS — PROJECT FINAL SUMMARY')
print('  AI-Powered Comparative Microbiome Analysis')
print('  Colorectal | Breast | Prostate Cancer')
print('=' * 78)

# ---- [1] Dataset sizes ------------------------------------
print('\n[1] DATASET SIZES')
if not meta_df.empty:
    if 'cancer_type' in meta_df.columns:
        counts = meta_df['cancer_type'].value_counts()  # count samples per cancer type
        for ct, n in counts.items():
            print(f'    {ct:<20s}: {n:>4d} samples')  # :<20 = left-align in 20 chars; :>4 = right-align in 4
    elif 'dataset' in meta_df.columns:
        counts = meta_df['dataset'].value_counts()
        for ct, n in counts.items():
            print(f'    {ct:<20s}: {n:>4d} samples')
    print(f'    Total           : {len(meta_df):>4d} samples')
else:
    print('    meta_combined.csv not found — run Notebook 02 first.')

# ---- [2] Machine Learning results -------------------------
print('\n[2] MACHINE LEARNING RESULTS')
if not model_comp_df.empty:
    # Flexibly find the AUC column (column names differ by run)
    auc_col = next((c for c in ['macro_auc', 'AUC', 'roc_auc', 'auc', 'CV_Macro_AUC']
                    if c in model_comp_df.columns), None)
    model_col = next((c for c in ['model', 'Model', 'classifier']
                      if c in model_comp_df.columns), None)
    if auc_col and model_col:
        best_row = model_comp_df.loc[model_comp_df[auc_col].idxmax()]  # row with highest AUC
        print(f'    Best model      : {best_row[model_col]}')
        print(f'    Macro-AUC       : {best_row[auc_col]:.4f}')
        for col in model_comp_df.columns:
            if col not in [model_col, auc_col]:
                try:
                    print(f'    {col:<20s}: {best_row[col]:.4f}')
                except Exception:
                    print(f'    {col:<20s}: {best_row[col]}')
    else:
        print(f'    Model comparison loaded: {model_comp_df.shape}')
        print(model_comp_df.to_string(index=False))
else:
    print('    model_comparison.csv not found — run Notebook 06 first.')

# ---- [3] Top 5 biomarkers ---------------------------------
print('\n[3] TOP 5 MICROBIAL BIOMARKERS (Composite Evidence Score)')
for i, row in biomarker_df.head(5).iterrows():
    print(f'    {i+1}. {row["genus"]:<25s} '
          f'score={row["composite_score"]:.3f}  '
          f'primary={str(row.get("primary_cancer","N/A")):<12s}  '
          f'role: {str(row["known_role"])[:50]}')  # truncate long role descriptions

# ---- [4] Pan-cancer signatures ----------------------------
print('\n[4] PAN-CANCER SIGNATURES')
if n_pancancer > 0:
    print(f'    {n_pancancer} genera enriched in both Colorectal and Breast cancer:')
    for g in pancancer_genera[:10]:  # show first 10 for readability
        print(f'      - {g}')
    if n_pancancer > 10:
        print(f'      ... and {n_pancancer - 10} more (see pancancer_shared_enriched_genera.csv)')
else:
    print('    Run Notebook 05 to generate pan-cancer signature results.')

# ---- [5] Diversity key findings ---------------------------
print('\n[5] KEY DIVERSITY FINDINGS')
alpha_path = os.path.join(RESULTS, 'alpha_diversity.csv')
if os.path.exists(alpha_path):
    alpha_df = pd.read_csv(alpha_path)
    print(f'    Alpha diversity file found: {alpha_df.shape}')
    if 'shannon' in alpha_df.columns and 'cancer_type' in alpha_df.columns:
        # Shannon diversity: measures how many different species are present and how evenly
        # they are distributed (like asking "how many different kinds of birds are in this forest?")
        sh = alpha_df.groupby('cancer_type')['shannon'].median()  # median per cancer type
        for ct, val in sh.items():
            print(f'    Median Shannon ({ct}): {val:.3f}')
else:
    print('    Alpha diversity summary not found. Run Notebook 03 for diversity analysis.')

# ---- [6] Output file status checklist ----------------------
print('\n[6] GENERATED OUTPUT FILES')
outputs = [
    os.path.join(FIGURES, 'fig01_study_design.png'),   # study design overview
    os.path.join(FIGURES, 'fig13_biomarker_panel.png'), # biomarker heatmap
    os.path.join(RESULTS, 'Table3_biomarker_panel.csv'),# final biomarker table
]
for f in outputs:
    status = 'OK' if os.path.exists(f) else 'MISSING'  # check if the file was successfully created
    print(f'    [{status}] {os.path.basename(f)}')

print('\n' + '=' * 78)
print('  Notebook 08 complete. Review figures and Table 3, then see Paper/draft_v1.md.')
print('=' * 78)